# Semantic Model Similarity

Compare every semantic model cataloged in the lakehouse and identify duplicates, near-duplicates, and subset relationships. Each model is reduced to a signature (tables, columns, measure names, measure definitions, DAX expressions, relationships, and data sources). Every pair gets two independent scores:

- a symmetric **composite similarity score** — structural Jaccard overlap plus text-embedding cosine similarity, tiered as duplicate / similar / distinct — that answers *"how alike are these two models overall?"*, and
- a directional **containment score** that answers *"does one model contain everything in the other?"*.

Duplicates are grouped into clusters. Results are shown in-notebook and written back to the lakehouse as Delta tables.


## Prerequisites

Run this notebook in a Fabric notebook runtime with a lakehouse attached — the same lakehouse the TOM catalog notebook wrote to. It must already contain the catalog Delta tables (`semantic_models`, `semantic_model_tables`, `semantic_model_columns`, `semantic_model_relationships`, `semantic_model_measures`, `semantic_model_datasources`). Text similarity uses a local scikit-learn TF-IDF vectorizer, so no external endpoint, key, or GPU/PyTorch runtime is required.

## Parameters

All tunable settings live here: the write mode, blocking toggle, tier thresholds, the ranked-table and heatmap knobs, and the per-signal weights. Adjust these, then run the notebook top to bottom. The catalog tables are read from the **attached lakehouse**, and results are written back to it.

In [ ]:
# Results are read from and written to the lakehouse attached to this notebook.
WRITE_MODE = "overwrite"  # "overwrite" replaces prior output; use "append" only if downstream supports it.

# Blocking limits comparisons to model pairs that share at least one table or measure name.
# Disable to force full pairwise comparison (slower on large catalogs).
ENABLE_BLOCKING = True

# Composite-score tier thresholds.
DUPLICATE_THRESHOLD = 0.95
SIMILAR_THRESHOLD = 0.70

# Containment threshold. Containment is a second, directional score that answers a
# different question than the composite similarity score: "does one model contain
# everything in the other?" rather than "how alike are the two models overall?".
# A pair whose stronger direction reaches this value is flagged as a containment
# candidate, and the two scores are reported side by side so you can filter on both.
CONTAINMENT_THRESHOLD = 0.95

# Report knobs.
TOP_N = 20  # Rows shown in the ranked pair table.
HEATMAP_MIN_SCORE = SIMILAR_THRESHOLD  # Hide heatmap cells scoring below this composite value.

# Relative weights for each similarity signal. Values are normalized, so they need not sum to 1.
SIMILARITY_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_dax_embedding": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Relative weights for each containment signal. Containment uses exact measure
# definitions (name + comment-stripped DAX) instead of the TF-IDF embedding, because
# lexical cosine similarity is symmetric and cannot establish that one model's measures
# are a subset of another's. Weights are normalized per direction over whichever signals
# the source model actually has.
CONTAINMENT_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_definitions": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Guard against misconfigured weights before any scoring runs.
for _weights_name, _weights in (
    ("SIMILARITY_WEIGHTS", SIMILARITY_WEIGHTS),
    ("CONTAINMENT_WEIGHTS", CONTAINMENT_WEIGHTS),
):
    if any(weight < 0 for weight in _weights.values()):
        raise ValueError(f"{_weights_name} must not contain negative weights.")
    if sum(_weights.values()) <= 0:
        raise ValueError(f"{_weights_name} must sum to a positive value.")


## Setup and computation

The code that builds the model signatures and scores every pair. Its input is hidden by default so you can focus on the parameters and results - click **Show input** on any cell to inspect it, or fold the whole section from this heading. You don't need to touch anything here.

In [ ]:
import itertools
import re
from collections import defaultdict

import numpy as np
import pandas as pd

### Load the catalog

Read the six catalog tables from the lakehouse into pandas. Missing tables are tolerated (they produce empty frames), so the run still completes for whichever signals are available.

In [ ]:
def load_delta(table_name):
    df = spark.read.format("delta").load('Tables/' + table_name)
    return df.toPandas()


models_df = load_delta("semantic_models")
tables_df = load_delta("semantic_model_tables")
columns_df = load_delta("semantic_model_columns")
relationships_df = load_delta("semantic_model_relationships")
measures_df = load_delta("semantic_model_measures")
datasources_df = load_delta("semantic_model_datasources")

if models_df.empty:
    raise ValueError(
        "No rows in the semantic_models table of the attached lakehouse. "
        "Run the TOM catalog notebook first."
    )

print(f"Models: {len(models_df)}")
print(
    f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
    f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
    f"Datasources: {len(datasources_df)}"
)

### Build model signatures

Each model is reduced to normalized sets (lowercased, whitespace-collapsed) for structural comparison, plus a text document (measure names + comment-stripped DAX) for embedding. Models with no measures fall back to their table and column names so the embedding document is never empty.

In [ ]:
def norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()


def norm_dax(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
    text = re.sub(r"//.*", " ", text)  # line comments
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def model_label(sig):
    return f"{sig['workspace_name']} / {sig['model_name']}"


signatures = {}
for _, row in models_df.iterrows():
    model_id = str(row["model_id"])
    signatures[model_id] = {
        "model_id": model_id,
        "workspace_id": str(row.get("workspace_id", "")),
        "workspace_name": str(row.get("workspace_name", "")),
        "model_name": str(row.get("model_name", "")),
        "tables": set(),
        "columns": set(),
        "measure_names": set(),
        "measure_definitions": set(),
        "relationships": set(),
        "datasources": set(),
        "dax_docs": [],
    }


def sig_for(model_id):
    return signatures.get(str(model_id))


for _, row in tables_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["tables"].add(norm(row["table_name"]))

for _, row in columns_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")

for _, row in measures_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        measure_name = norm(row["measure_name"])
        measure_dax = norm_dax(row.get("expression"))
        sig["measure_names"].add(measure_name)
        # Name + DAX key so containment only credits measures whose logic also matches.
        sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
        sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())

for _, row in relationships_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        key = (
            f"{norm(row['from_table'])}.{norm(row['from_column'])}"
            f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
        )
        sig["relationships"].add(key)

for _, row in datasources_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
        conn_norm = norm(conn)
        if conn_norm:
            sig["datasources"].add(conn_norm)

# Build the embedding document per model, with a structural fallback when no measures exist.
for sig in signatures.values():
    parts = list(sig["dax_docs"])
    if not parts:
        parts = sorted(sig["tables"]) + sorted(sig["columns"])
    sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])

model_ids = list(signatures.keys())
print(f"Built signatures for {len(model_ids)} models.")


### Candidate pairs and structural similarity

With blocking enabled, only model pairs that share at least one table or measure name are scored, which avoids a full O(n²) comparison on large catalogs. Jaccard overlap is computed per signal for each candidate pair.

In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union else 0.0


def coverage(source_set, other_set):
    # Directional: the fraction of source_set's members that also appear in other_set.
    # Returns None when the signal is absent from the source, so it can be excluded from
    # the weighted average rather than counted as a spurious perfect match.
    if not source_set:
        return None
    return len(source_set & other_set) / len(source_set)


def weighted_containment(source_sig, other_sig, weights):
    # How completely source_sig is contained in other_sig: a weighted mean of the
    # per-signal coverages, normalized over whichever signals the source actually has.
    accumulated = 0.0
    total_weight = 0.0
    for signal, weight in weights.items():
        signal_coverage = coverage(source_sig[signal], other_sig[signal])
        if signal_coverage is None:
            continue
        accumulated += weight * signal_coverage
        total_weight += weight
    return accumulated / total_weight if total_weight else 0.0


def classify_containment(a_in_b, b_in_a, threshold):
    a_contained = a_in_b >= threshold
    b_contained = b_in_a >= threshold
    if a_contained and b_contained:
        return "equivalent"
    if b_contained:
        return "model_a_contains_model_b"
    if a_contained:
        return "model_b_contains_model_a"
    return "partial_overlap"


if ENABLE_BLOCKING:
    block_index = defaultdict(set)
    for model_id, sig in signatures.items():
        for table_name in sig["tables"]:
            block_index[("t", table_name)].add(model_id)
        for measure_name in sig["measure_names"]:
            block_index[("m", measure_name)].add(model_id)
    candidate_pairs = set()
    for group in block_index.values():
        if len(group) > 1:
            candidate_pairs.update(itertools.combinations(sorted(group), 2))
else:
    candidate_pairs = set(itertools.combinations(sorted(model_ids), 2))

print(f"Candidate pairs to score: {len(candidate_pairs)}")


### Semantic embeddings

Encode each model's document once as a TF-IDF vector over its DAX / name tokens (no PyTorch dependency), then derive cosine similarity from the L2-normalized vectors (a dot product) for each candidate pair.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF avoids the torch/transformers dependency chain (the Fabric runtime ships
# an older PyTorch than recent transformers require). Rows are L2-normalized, so
# cosine similarity stays a plain dot product for the downstream scoring step.
docs = [signatures[model_id]["doc"] for model_id in model_ids]
vectorizer = TfidfVectorizer(min_df=1, norm="l2")

embeddings = vectorizer.fit_transform(docs).toarray()
print(f"Encoded {len(docs)} model documents into {embeddings.shape[1]}-dim TF-IDF vectors.")
embedding_index = {model_id: idx for idx, model_id in enumerate(model_ids)}

### Score the models

Each candidate pair gets two independent scores:

- **Composite score** — a weighted blend of the six similarity signals that answers *"how alike are these two models overall?"*. It is symmetric, drives the duplicate / similar / distinct tiers, and duplicate-tier pairs are merged into clusters with union-find.
- **Containment score** — a directional measure that answers *"does one model contain everything in the other?"*. It is the stronger of the two directional coverages (A-in-B and B-in-A), and `containment_relationship` records which model contains which. A small model fully absorbed by a much larger one scores high here even when the composite score is only moderate.

Both scores are written to every pair row so you can filter on either one.


In [ ]:
weight_sum = sum(SIMILARITY_WEIGHTS.values())
pair_rows = []

for model_id_a, model_id_b in candidate_pairs:
    sig_a = signatures[model_id_a]
    sig_b = signatures[model_id_b]

    j_tables = jaccard(sig_a["tables"], sig_b["tables"])
    j_columns = jaccard(sig_a["columns"], sig_b["columns"])
    j_measures = jaccard(sig_a["measure_names"], sig_b["measure_names"])
    j_relationships = jaccard(sig_a["relationships"], sig_b["relationships"])
    j_datasources = jaccard(sig_a["datasources"], sig_b["datasources"])
    cosine = float(
        np.dot(embeddings[embedding_index[model_id_a]], embeddings[embedding_index[model_id_b]])
    )
    cosine = max(0.0, min(1.0, cosine))

    composite = (
        SIMILARITY_WEIGHTS["tables"] * j_tables
        + SIMILARITY_WEIGHTS["columns"] * j_columns
        + SIMILARITY_WEIGHTS["measure_names"] * j_measures
        + SIMILARITY_WEIGHTS["measure_dax_embedding"] * cosine
        + SIMILARITY_WEIGHTS["relationships"] * j_relationships
        + SIMILARITY_WEIGHTS["datasources"] * j_datasources
    ) / weight_sum

    # Directional containment: how much of each model is absorbed by the other.
    a_in_b = weighted_containment(sig_a, sig_b, CONTAINMENT_WEIGHTS)
    b_in_a = weighted_containment(sig_b, sig_a, CONTAINMENT_WEIGHTS)
    containment_score = max(a_in_b, b_in_a)
    containment_relationship = classify_containment(a_in_b, b_in_a, CONTAINMENT_THRESHOLD)

    if composite >= DUPLICATE_THRESHOLD:
        tier = "duplicate"
    elif composite >= SIMILAR_THRESHOLD:
        tier = "similar"
    else:
        tier = "distinct"

    pair_rows.append({
        "model_id_a": model_id_a,
        "model_a": model_label(sig_a),
        "workspace_a": sig_a["workspace_name"],
        "model_id_b": model_id_b,
        "model_b": model_label(sig_b),
        "workspace_b": sig_b["workspace_name"],
        "same_model_name": norm(sig_a["model_name"]) == norm(sig_b["model_name"]),
        "cross_workspace": sig_a["workspace_id"] != sig_b["workspace_id"],
        "jaccard_tables": round(j_tables, 4),
        "jaccard_columns": round(j_columns, 4),
        "jaccard_measure_names": round(j_measures, 4),
        "jaccard_relationships": round(j_relationships, 4),
        "jaccard_datasources": round(j_datasources, 4),
        "dax_embedding_cosine": round(cosine, 4),
        "composite_score": round(composite, 4),
        "containment_score": round(containment_score, 4),
        "containment_relationship": containment_relationship,
        "model_a_in_model_b": round(a_in_b, 4),
        "model_b_in_model_a": round(b_in_a, 4),
        "tier": tier,
    })

pairs_df = pd.DataFrame(pair_rows)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values("composite_score", ascending=False).reset_index(drop=True)

# Union-find clustering over duplicate-tier pairs.
parent = {model_id: model_id for model_id in model_ids}


def find(node):
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = parent[node]
    return node


def union(node_a, node_b):
    root_a, root_b = find(node_a), find(node_b)
    if root_a != root_b:
        parent[root_a] = root_b


if not pairs_df.empty:
    for _, row in pairs_df[pairs_df["tier"] == "duplicate"].iterrows():
        union(row["model_id_a"], row["model_id_b"])

cluster_members = defaultdict(list)
for model_id in model_ids:
    cluster_members[find(model_id)].append(model_id)

cluster_rows = []
cluster_number = 0
for members in cluster_members.values():
    if len(members) > 1:
        cluster_number += 1
        for model_id in members:
            sig = signatures[model_id]
            cluster_rows.append({
                "cluster_id": cluster_number,
                "cluster_size": len(members),
                "model_id": model_id,
                "model": model_label(sig),
                "workspace_name": sig["workspace_name"],
                "model_name": sig["model_name"],
            })

clusters_df = pd.DataFrame(cluster_rows)

# Per-model signature summary.
signature_rows = []
for sig in signatures.values():
    signature_rows.append({
        "model_id": sig["model_id"],
        "workspace_name": sig["workspace_name"],
        "model_name": sig["model_name"],
        "table_count": len(sig["tables"]),
        "column_count": len(sig["columns"]),
        "measure_count": len(sig["measure_names"]),
        "relationship_count": len(sig["relationships"]),
        "datasource_count": len(sig["datasources"]),
    })
signatures_df = pd.DataFrame(signature_rows)

duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
containment_count = (
    int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
)
print(f"Duplicate pairs: {duplicate_count} | Similar pairs: {similar_count} | Containment pairs: {containment_count}")
print(f"Duplicate clusters: {cluster_number}")


## Overview

A snapshot of the strongest consolidation candidates found in this run: headline counts, the top model pairs to review, duplicate clusters, and subset (containment) relationships. Everything here is derived from the scored results — expand **Details &amp; exploration** below for the full ranked tables and the similarity / containment heatmaps.

In [ ]:
# Overview panel: a presentation-first summary of the strongest consolidation candidates,
# rendered from the scored frames. Input is hidden in Fabric so readers see only the panel.
import html

OVERVIEW_TOP_N = 5  # candidates surfaced in the hero; the full ranked list is in the details below.

_OVERVIEW_CSS = """<style>
.sms-ov{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;color:#1d1d1f;background:#f5f5f7;padding:28px 28px 30px;border-radius:20px;max-width:1080px;margin:8px auto;-webkit-font-smoothing:antialiased;}
.sms-ov *{box-sizing:border-box;}
.sms-ov .eyebrow{font-size:12px;font-weight:600;letter-spacing:.07em;text-transform:uppercase;color:#8a8a8e;margin:0 0 8px;}
.sms-ov .head{font-size:25px;line-height:1.28;font-weight:600;margin:0 0 8px;letter-spacing:-.01em;}
.sms-ov .sub{font-size:14px;color:#6e6e73;margin:0 0 22px;line-height:1.5;}
.sms-ov .kpis{display:flex;flex-wrap:wrap;gap:14px;margin-bottom:8px;}
.sms-ov .kpi{flex:1 1 150px;background:#fff;border-radius:16px;padding:18px 20px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
.sms-ov .kpi .n{font-size:34px;font-weight:600;letter-spacing:-.02em;line-height:1;}
.sms-ov .kpi .l{font-size:12.5px;color:#6e6e73;margin-top:7px;}
.sms-ov .kpi.accent .n{color:#0071e3;}
.sms-ov .kpi.warn .n{color:#c9330a;}
.sms-ov h3.sec{font-size:18px;font-weight:600;margin:28px 0 14px;letter-spacing:-.01em;}
.sms-ov .card{background:#fff;border-radius:16px;padding:16px 18px;box-shadow:0 1px 3px rgba(0,0,0,.06);margin-bottom:12px;}
.sms-ov .pair{display:flex;align-items:center;gap:16px;flex-wrap:wrap;}
.sms-ov .models{flex:1 1 380px;min-width:240px;}
.sms-ov .mname{font-size:15px;font-weight:600;}
.sms-ov .wname{font-size:12px;color:#8a8a8e;}
.sms-ov .vs{color:#c7c7cc;margin:0 8px;}
.sms-ov .scorewrap{flex:0 0 190px;}
.sms-ov .track{height:8px;background:#e9e9eb;border-radius:6px;overflow:hidden;}
.sms-ov .fill{height:100%;border-radius:6px;background:#0071e3;}
.sms-ov .scoreval{font-size:12px;color:#6e6e73;margin-top:5px;}
.sms-ov .pill{display:inline-block;font-size:11.5px;font-weight:600;padding:3px 10px;border-radius:999px;}
.sms-ov .pill.dup{background:#ffe9e3;color:#c9330a;}
.sms-ov .pill.sim{background:#fff2df;color:#96590a;}
.sms-ov .chip{display:inline-block;font-size:11px;font-weight:500;padding:2px 9px;border-radius:999px;background:#eef1f6;color:#5b5b60;margin-left:6px;}
.sms-ov .chip.xws{background:#efe9fc;color:#6b3fd4;}
.sms-ov .verdict{font-size:13.5px;color:#3a3a3c;margin-top:11px;line-height:1.45;}
.sms-ov details{margin-top:11px;}
.sms-ov summary{font-size:12.5px;color:#0071e3;cursor:pointer;list-style:none;}
.sms-ov summary::-webkit-details-marker{display:none;}
.sms-ov .evid{display:flex;flex-wrap:wrap;gap:8px 18px;margin-top:11px;font-size:12.5px;color:#6e6e73;}
.sms-ov .evid b{color:#1d1d1f;font-weight:600;}
.sms-ov .members{margin:10px 0 0;padding:0;list-style:none;}
.sms-ov .members li{display:flex;justify-content:space-between;align-items:baseline;padding:7px 0;border-top:1px solid #f0f0f2;}
.sms-ov .members li:first-child{border-top:none;}
.sms-ov .members .w{color:#8a8a8e;font-size:12px;}
.sms-ov .badge{font-size:11.5px;font-weight:600;color:#0071e3;background:#eef4ff;border-radius:999px;padding:3px 11px;}
.sms-ov .empty{background:#fff;border-radius:16px;padding:26px;text-align:center;color:#6e6e73;font-size:14px;box-shadow:0 1px 3px rgba(0,0,0,.06);}
.sms-ov .legend{font-size:12px;color:#8a8a8e;margin-top:24px;line-height:1.7;}
.sms-ov .legend b{color:#3a3a3c;}
</style>"""


def _esc(value):
    return html.escape("" if value is None else str(value))


def _bar(score, color="#0071e3", label="composite"):
    width = max(0.0, min(1.0, float(score))) * 100
    return (
        f'<div class="track"><div class="fill" style="width:{width:.1f}%;background:{color};"></div></div>'
        f'<div class="scoreval">{float(score):.2f} {label}</div>'
    )


def _verdict(row):
    tier = row["tier"]
    rel = row.get("containment_relationship", "")
    contained = float(row.get("containment_score", 0.0)) >= CONTAINMENT_THRESHOLD
    if tier == "duplicate" and bool(row.get("cross_workspace")):
        return "Very likely the same model living in two workspaces &mdash; consolidate to a single source."
    if tier == "duplicate" and bool(row.get("same_model_name")):
        return "Duplicate models sharing the same name &mdash; consolidate."
    if tier == "duplicate":
        return "Near-identical structure &mdash; a strong consolidation candidate."
    if contained and rel in ("model_a_contains_model_b", "model_b_contains_model_a"):
        return "One model is effectively a subset of the other &mdash; consider retiring the smaller one."
    return "Substantial overlap &mdash; review for shared logic or consolidation."


total_models = int(len(signatures_df)) if not signatures_df.empty else len(signatures)

if not pairs_df.empty:
    _flagged = pairs_df[pairs_df["tier"].isin(["duplicate", "similar"])].sort_values(
        "composite_score", ascending=False
    ).reset_index(drop=True)
    _contain = pairs_df[pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD].sort_values(
        "containment_score", ascending=False
    ).reset_index(drop=True)
else:
    _flagged = pairs_df
    _contain = pairs_df

# Headline sentence.
if _flagged.empty and clusters_df.empty and _contain.empty:
    _headline = f"No duplicates or near-duplicates found across {total_models} models at the current thresholds."
else:
    _parts = []
    if duplicate_count:
        _parts.append(f"{duplicate_count} duplicate")
    if similar_count:
        _parts.append(f"{similar_count} near-duplicate")
    if _parts:
        _total_pairs = duplicate_count + similar_count
        _lead = " and ".join(_parts) + (" pairs" if _total_pairs != 1 else " pair")
    else:
        _lead = "overlapping models"
    _tail = ""
    if containment_count:
        _tail += f", plus {containment_count} subset relationship" + ("s" if containment_count != 1 else "")
    if cluster_number:
        _tail += f", forming {cluster_number} duplicate cluster" + ("s" if cluster_number != 1 else "")
    _headline = f"Across {total_models} models, found {_lead}{_tail}."

# KPI cards.
_kpi_defs = [
    ("accent", total_models, "Models scanned"),
    ("warn" if duplicate_count else "", duplicate_count, "Duplicate pairs"),
    ("", similar_count, "Similar pairs"),
    ("", containment_count, "Containment candidates"),
    ("", cluster_number, "Duplicate clusters"),
]
_kpi_html = "".join(
    f'<div class="kpi {cls}"><div class="n">{val}</div><div class="l">{lbl}</div></div>'
    for cls, val, lbl in _kpi_defs
)

# Top consolidation candidates.
if _flagged.empty:
    _cand_html = ('<div class="empty">No duplicate or near-duplicate pairs at the current thresholds. '
                  'Lower DUPLICATE_THRESHOLD / SIMILAR_THRESHOLD in Parameters to widen the search.</div>')
else:
    _cards = []
    for _, r in _flagged.head(OVERVIEW_TOP_N).iterrows():
        _is_dup = r["tier"] == "duplicate"
        _pill = f'<span class="pill {"dup" if _is_dup else "sim"}">{"Duplicate" if _is_dup else "Similar"}</span>'
        _xws = '<span class="chip xws">cross-workspace</span>' if bool(r.get("cross_workspace")) else ""
        _evid = (
            '<div class="evid">'
            f'<span>Tables <b>{r["jaccard_tables"]:.2f}</b></span>'
            f'<span>Columns <b>{r["jaccard_columns"]:.2f}</b></span>'
            f'<span>Measures <b>{r["jaccard_measure_names"]:.2f}</b></span>'
            f'<span>Relationships <b>{r["jaccard_relationships"]:.2f}</b></span>'
            f'<span>Data sources <b>{r["jaccard_datasources"]:.2f}</b></span>'
            f'<span>DAX text <b>{r["dax_embedding_cosine"]:.2f}</b></span>'
            f'<span>Containment <b>{r["containment_score"]:.2f}</b></span>'
            '</div>'
        )
        _cards.append(
            '<div class="card"><div class="pair"><div class="models">'
            f'{_pill}{_xws}'
            f'<div style="margin-top:9px;"><span class="mname">{_esc(r["model_a"])}</span> '
            f'<span class="wname">&middot; {_esc(r["workspace_a"])}</span>'
            '<span class="vs">&#8596;</span>'
            f'<span class="mname">{_esc(r["model_b"])}</span> '
            f'<span class="wname">&middot; {_esc(r["workspace_b"])}</span></div>'
            '</div>'
            f'<div class="scorewrap">{_bar(r["composite_score"])}</div>'
            '</div>'
            f'<div class="verdict">{_verdict(r)}</div>'
            f'<details><summary>Why this scored high</summary>{_evid}</details>'
            '</div>'
        )
    _cand_html = "".join(_cards)

# Duplicate clusters.
if clusters_df.empty:
    _cluster_html = ""
else:
    _blocks = []
    for _cid, _grp in clusters_df.sort_values(["cluster_id", "model"]).groupby("cluster_id"):
        _members = "".join(
            f'<li><span class="mname" style="font-weight:600;font-size:13.5px;">{_esc(m["model_name"])}</span>'
            f'<span class="w">{_esc(m["workspace_name"])}</span></li>'
            for _, m in _grp.iterrows()
        )
        _size = int(_grp["cluster_size"].iloc[0])
        _blocks.append(
            '<div class="card"><div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<div class="mname">Cluster {int(_cid)}</div><span class="badge">{_size} models</span></div>'
            f'<ul class="members">{_members}</ul></div>'
        )
    _cluster_html = '<h3 class="sec">Duplicate clusters</h3>' + "".join(_blocks)

# Containment highlights.
if _contain.empty:
    _contain_html = ""
else:
    _rows = []
    for _, r in _contain.head(OVERVIEW_TOP_N).iterrows():
        a_in_b = float(r["model_a_in_model_b"])
        b_in_a = float(r["model_b_in_model_a"])
        rel = r.get("containment_relationship", "")
        if rel == "equivalent":
            _desc = f'<span class="mname">{_esc(r["model_a"])}</span> <span class="vs">&#8801;</span> <span class="mname">{_esc(r["model_b"])}</span>'
            _cov = max(a_in_b, b_in_a)
            _note = "Equivalent &mdash; each model contains the other."
        elif a_in_b >= b_in_a:
            _desc = f'<span class="mname">{_esc(r["model_a"])}</span> <span class="vs">&#8834;</span> <span class="mname">{_esc(r["model_b"])}</span>'
            _cov = a_in_b
            _note = f'{_esc(r["model_a"])} is contained in {_esc(r["model_b"])}.'
        else:
            _desc = f'<span class="mname">{_esc(r["model_b"])}</span> <span class="vs">&#8834;</span> <span class="mname">{_esc(r["model_a"])}</span>'
            _cov = b_in_a
            _note = f'{_esc(r["model_b"])} is contained in {_esc(r["model_a"])}.'
        _rows.append(
            '<div class="card"><div class="pair"><div class="models">'
            f'{_desc}<div class="verdict" style="margin-top:6px;">{_note}</div></div>'
            f'<div class="scorewrap">{_bar(_cov, color="#0058c9", label="coverage")}</div>'
            '</div></div>'
        )
    _contain_html = '<h3 class="sec">Containment highlights</h3>' + "".join(_rows)

_legend = (
    '<div class="legend"><b>Duplicate</b> &ge; '
    f'{DUPLICATE_THRESHOLD:.2f} composite &middot; <b>Similar</b> &ge; {SIMILAR_THRESHOLD:.2f} '
    f'&middot; <b>Containment</b> &ge; {CONTAINMENT_THRESHOLD:.2f} coverage. '
    'Scores are metadata-based (tables, columns, measures &amp; DAX, relationships, data sources) '
    '&mdash; confirm the models serve the same purpose before consolidating.</div>'
)

_overview_html = (
    _OVERVIEW_CSS
    + '<div class="sms-ov">'
    + '<div class="eyebrow">Semantic Model Similarity</div>'
    + f'<div class="head">{_headline}</div>'
    + '<div class="sub">Start here &mdash; the strongest consolidation candidates from this run. '
      'Full ranked tables and heatmaps are in "Details &amp; exploration" below.</div>'
    + f'<div class="kpis">{_kpi_html}</div>'
    + '<h3 class="sec">Top consolidation candidates</h3>'
    + _cand_html
    + _cluster_html
    + _contain_html
    + _legend
    + '</div>'
)

displayHTML(_overview_html)


## Details &amp; exploration

The full ranked tables and interactive heatmaps behind the overview above. Each model pair carries two scores: a symmetric **composite similarity** score (how alike two models are overall) and a directional **containment** score (whether one model contains everything in the other).

### Ranked model pairs

The highest-scoring flagged pairs (duplicate + similar), the duplicate clusters, and a similarity heatmap of the models that appear in at least one flagged pair.

**How to read the heatmap:** each cell is the composite similarity (0–1) between two models. Only the upper triangle is shown (the matrix is symmetric and the diagonal is self-similarity), and cells scoring below `HEATMAP_MIN_SCORE` are left blank. Rows and columns are grouped by clustering, so near-duplicate models sit next to each other.

In [ ]:
# Flagged pairs (duplicate + similar) are the canonical ranked output, reused by the heatmap below.
if pairs_df.empty:
    print("No candidate pairs were produced.")
    flagged_df = pairs_df
else:
    flagged_df = pairs_df[pairs_df["tier"].isin(["duplicate", "similar"])].reset_index(drop=True)

if flagged_df.empty:
    print("No duplicate or similar pairs were found at the current thresholds.")
else:
    ranked = flagged_df.copy()
    ranked.insert(0, "rank", range(1, len(ranked) + 1))
    column_order = [
        "rank", "tier", "composite_score",
        "containment_score", "containment_relationship",
        "model_a", "workspace_a", "model_b", "workspace_b",
        "model_a_in_model_b", "model_b_in_model_a",
        "same_model_name", "cross_workspace",
        "jaccard_tables", "jaccard_columns", "jaccard_measure_names",
        "jaccard_relationships", "jaccard_datasources", "dax_embedding_cosine",
    ]
    ranked = ranked[[column for column in column_order if column in ranked.columns]]

    print(f"Flagged pairs: {len(flagged_df)} of {len(pairs_df)} scored pairs (showing top {min(TOP_N, len(ranked))}).")
    display(ranked.head(TOP_N))

    scores = flagged_df["composite_score"]
    print(
        "Composite score across flagged pairs -> "
        f"max {scores.max():.4f} | median {scores.median():.4f} | min {scores.min():.4f}"
    )


### Containment candidates

Containment is directional: it measures how completely one model's objects are a subset of the other's, independent of overall similarity. A small model fully absorbed by a much larger one scores high here even when the composite similarity is only moderate — the exact case the symmetric composite score understates.

Pairs are filtered to those whose stronger direction reaches `CONTAINMENT_THRESHOLD`, ranked by containment strength. `containment_relationship` names which model contains which, and `model_a_in_model_b` / `model_b_in_model_a` show the two directional coverages behind the score.


In [ ]:
# Containment-candidate report: pairs where one model effectively contains the other,
# ranked by containment strength. This is independent of the composite similarity tier,
# so it surfaces subset relationships (a small model fully absorbed by a larger one) that
# the similarity-ranked table above can miss.
if pairs_df.empty:
    print("No candidate pairs were produced.")
else:
    contained_df = pairs_df[pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD].copy()
    if contained_df.empty:
        print(f"No containment candidates at CONTAINMENT_THRESHOLD = {CONTAINMENT_THRESHOLD:.2f}.")
    else:
        contained_df = contained_df.sort_values(
            ["containment_score", "composite_score"], ascending=False
        ).reset_index(drop=True)
        contained_df.insert(0, "rank", range(1, len(contained_df) + 1))
        containment_columns = [
            "rank", "containment_relationship", "containment_score", "composite_score",
            "model_a", "workspace_a", "model_b", "workspace_b",
            "model_a_in_model_b", "model_b_in_model_a",
            "same_model_name", "cross_workspace",
        ]
        containment_columns = [column for column in containment_columns if column in contained_df.columns]

        print(f"Containment candidates: {len(contained_df)} (showing top {min(TOP_N, len(contained_df))}).")
        display(contained_df[containment_columns].head(TOP_N))


### Duplicate clusters

In [ ]:
if clusters_df.empty:
    print("No duplicate clusters were found.")
else:
    display(clusters_df.sort_values(["cluster_id", "model"]))

### Composite similarity heatmap

In [ ]:
# Heatmap of composite similarity across models that appear in at least one flagged pair.
# Built from flagged_df, so the scores here match the ranked table and the persisted output.
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

_HEATMAP_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif'

if flagged_df.empty:
    print("No flagged pairs to plot.")
else:
    involved_ids = sorted(
        set(flagged_df["model_id_a"]) | set(flagged_df["model_id_b"]),
        key=lambda model_id: model_label(signatures[model_id]),
    )
    labels = [model_label(signatures[model_id]) for model_id in involved_ids]
    position = {model_id: index for index, model_id in enumerate(involved_ids)}
    count = len(involved_ids)

    matrix = np.eye(count)
    for _, row in flagged_df.iterrows():
        i, j = position[row["model_id_a"]], position[row["model_id_b"]]
        matrix[i, j] = matrix[j, i] = row["composite_score"]

    # Group similar models together so near-duplicates sit next to each other.
    order = list(range(count))
    if count > 2:
        try:
            linkage_matrix = linkage(squareform(1.0 - matrix, checks=False), method="ward")
            order = dendrogram(linkage_matrix, no_plot=True)["leaves"]
        except Exception as error:
            print(f"Clustering failed ({error}); using catalog order.")
    ordered_labels = [labels[index] for index in order]
    ordered_matrix = matrix[np.ix_(order, order)]

    # Upper triangle only: the matrix is symmetric and the diagonal is self-similarity.
    upper = ordered_matrix.astype(float)
    for i in range(count):
        for j in range(i + 1):
            upper[i, j] = np.nan
    upper[upper < HEATMAP_MIN_SCORE] = np.nan

    figure = go.Figure(
        data=go.Heatmap(
            z=upper,
            x=ordered_labels,
            y=ordered_labels,
            colorscale="Viridis",
            zmin=0.0,
            zmax=1.0,
            xgap=1,
            ygap=1,
            colorbar={"title": "Composite"},
            hovertemplate="Source: %{y}<br>Target: %{x}<br>Composite: %{z:.4f}<extra></extra>",
        )
    )
    figure.update_layout(
        title=f"Composite similarity of flagged models (score >= {HEATMAP_MIN_SCORE:.2f})",
        height=max(500, min(1400, count * 40 + 300)),
        width=max(650, min(1600, count * 40 + 420)),
        xaxis={"tickangle": 45},
        yaxis={"autorange": "reversed"},
        margin={"l": 220, "r": 80, "t": 90, "b": 220},
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        font={"family": _HEATMAP_FONT, "color": "#1d1d1f", "size": 12},
        title_font={"size": 15, "color": "#1d1d1f"},
    )
    displayHTML(figure.to_html(include_plotlyjs="inline", full_html=False))


### Containment matrix

Unlike the symmetric composite heatmap above, containment is directional, so this is a **full square** matrix. Each cell is the fraction of the **row** model's objects that are found in the **column** model — how completely the row model is *contained in* the column model. The diagonal is 1.0 (every model contains itself).

- Read **across a row** to see which models absorb it — a bright cell means that column model is a superset.
- Read **down a column** to see everything that model absorbs.
- A bright off-diagonal cell whose mirror is dark is a clear subset/superset relationship: the row model is contained in the column model, but not the reverse.

Only models that appear in at least one containment candidate (stronger direction ≥ `CONTAINMENT_THRESHOLD`) are shown.


In [ ]:
# Directional containment matrix across models that appear in at least one containment candidate.
# Unlike the composite heatmap this is a FULL square: matrix[i, j] = fraction of the row model i
# contained in the column model j (model_i_in_model_j), so the two triangles differ.
import plotly.graph_objects as go

_HEATMAP_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif'

if pairs_df.empty:
    print("No candidate pairs were produced.")
else:
    contained_pairs = pairs_df[pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD]
    if contained_pairs.empty:
        print(f"No containment candidates at CONTAINMENT_THRESHOLD = {CONTAINMENT_THRESHOLD:.2f}; nothing to plot.")
    else:
        involved_ids = sorted(
            set(contained_pairs["model_id_a"]) | set(contained_pairs["model_id_b"]),
            key=lambda model_id: model_label(signatures[model_id]),
        )
        labels = [model_label(signatures[model_id]) for model_id in involved_ids]
        position = {model_id: index for index, model_id in enumerate(involved_ids)}
        involved_set = set(involved_ids)
        count = len(involved_ids)

        # Diagonal is 1.0 (every model fully contains itself); off-diagonals are directional.
        matrix = np.eye(count)
        for _, row in pairs_df.iterrows():
            if row["model_id_a"] in involved_set and row["model_id_b"] in involved_set:
                i, j = position[row["model_id_a"]], position[row["model_id_b"]]
                matrix[i, j] = row["model_a_in_model_b"]  # row A contained in column B
                matrix[j, i] = row["model_b_in_model_a"]  # row B contained in column A

        figure = go.Figure(
            data=go.Heatmap(
                z=matrix,
                x=labels,
                y=labels,
                colorscale="Cividis",  # distinct from the Viridis composite heatmap; colourblind-safe
                zmin=0.0,
                zmax=1.0,
                xgap=1,
                ygap=1,
                colorbar={"title": "Coverage"},
                hovertemplate="Contained model: %{y}<br>Container model: %{x}<br>Coverage: %{z:.4f}<extra></extra>",
            )
        )
        figure.update_layout(
            title=f"Directional containment: row model contained in column model (candidates >= {CONTAINMENT_THRESHOLD:.2f})",
            height=max(500, min(1400, count * 40 + 300)),
            width=max(650, min(1600, count * 40 + 420)),
            xaxis={"tickangle": 45},
            yaxis={"autorange": "reversed"},
            margin={"l": 220, "r": 80, "t": 90, "b": 220},
            paper_bgcolor="#ffffff",
            plot_bgcolor="#ffffff",
            font={"family": _HEATMAP_FONT, "color": "#1d1d1f", "size": 12},
            title_font={"size": 15, "color": "#1d1d1f"},
        )
        displayHTML(figure.to_html(include_plotlyjs="inline", full_html=False))


## Save results to the lakehouse

Write the per-model signature summary, the full pairwise scores, and the duplicate clusters as Delta tables so the analysis is queryable outside this notebook.

In [ ]:
similarity_outputs = {
    "semantic_model_signatures": signatures_df,
    "semantic_model_similarity_pairs": pairs_df,
    "semantic_model_duplicate_clusters": clusters_df,
}

for table_name, frame in similarity_outputs.items():
    if frame.empty:
        print(f"Skipped {table_name}: no rows")
        continue
    spark.createDataFrame(frame).write.format("delta").mode(WRITE_MODE).option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

## Next steps

- Start with the **duplicate clusters** and the top of the **ranked table** — these are the strongest consolidation candidates. `cross_workspace` pairs often indicate the same model copied between workspaces.
- Use the **containment candidates** table to find subset relationships: `model_a_contains_model_b` (or the reverse) means one model holds everything in the other plus more — a superseding model or an extract that may be retired. `semantic_model_similarity_pairs` carries both `composite_score` and `containment_score`, so you can filter on either independently (for example, high containment with only moderate similarity = a small model absorbed by a much larger one).
- Similarity and containment are metadata-based. Before acting on a pair, confirm the models truly serve the same purpose; they do not compare report layouts, row-level data, refresh history, or security roles.
- If the tiers look too strict or too loose, calibrate `DUPLICATE_THRESHOLD`, `SIMILAR_THRESHOLD`, `CONTAINMENT_THRESHOLD`, `SIMILARITY_WEIGHTS`, and `CONTAINMENT_WEIGHTS` in the Configuration cell against a few known pairs, then re-run.
- The three output tables (`semantic_model_signatures`, `semantic_model_similarity_pairs`, `semantic_model_duplicate_clusters`) are in the attached lakehouse for querying outside this notebook.
